# **Awful Sorting**
Análisis del Bogo-sort y distribuciones geométricas.

## **Teorema 2 — Comparaciones esperadas**

Análisis del costo de detectar que un arreglo está ordenado en bogo-sort.

### ***Enunciado del problema · 6a***

**Explique la fórmula usada para calcular P[Iₖ] en la prueba del Teorema 2 del paper "Sorting the Slow Way".**

En el análisis del algoritmo Bogo-sort, la verificación para saber si el arreglo está ordenado consiste en comparar pares adyacentes hasta encontrar uno que esté fuera de orden o llegar al final.

Definimos $I_k$ como el evento en el que "se necesitan al menos $k$ comparaciones para verificar el orden" (es decir, las primeras $k-1$ comparaciones no detectaron ningún problema porque esos elementos estaban ordenados). 

La fórmula para calcular esta probabilidad es:
$$P[I_k] = \frac{1}{k!}$$

**Justificación:** Para que pasemos las primeras $k$ comparaciones sin fallar, necesitamos que esos elementos específicos estén en orden. Si tomamos esos elementos, existen $k!$ permutaciones (ordenamientos) posibles para ellos. Dado que asumimos una permutación inicial uniformemente aleatoria, todas esas $k!$ permutaciones son igualmente probables. Sin embargo, solo **una** de esas permutaciones representa el ordenamiento correcto (ascendente). Por lo tanto, la probabilidad de que esos elementos estén ordenados por puro azar es exactamente $\frac{1}{k!}$.

*(Nota: Es un error común pensar que cada comparación de pares es un evento independiente con probabilidad $1/2$, lo que daría $(1/2)^{k-1}$. Las comparaciones no son independientes debido a la propiedad transitiva del orden, de ahí que la probabilidad real dependa de las permutaciones totales del subgrupo, $k!$).*

### **Enunciado del Problema 6B**
**En la prueba del Teorema 2, ¿por qué $E[C] = \sum_{k \ge 1} P[I_k]$?**

Esta igualdad se basa en una identidad algebraica estándar en teoría de probabilidad conocida como la **fórmula de la suma de la cola (Tail sum formula)** para el valor esperado de una variable aleatoria entera no negativa.

Sea $C$ el número de comparaciones necesarias para detectar que el arreglo NO está ordenado. $C$ es una variable aleatoria entera no negativa. Por definición, la esperanza matemática de una variable de este tipo se puede expresar como la suma de las probabilidades de que la variable sea mayor o igual a $k$:
$$E[C] = \sum_{k \ge 1} P(C \ge k)$$

Como definimos anteriormente, $I_k$ es exactamente el evento en el que se necesitan al menos $k$ comparaciones, lo que equivale a decir que $C \ge k$. Sustituyendo esto, obtenemos:
$$E[C] = \sum_{k \ge 1} P[I_k]$$

Sustituyendo la probabilidad que encontramos en el inciso 6A, la sumatoria queda como:
$$E[C] = \sum_{k=1}^{n-1} \frac{1}{k!}$$

A medida que $n$ crece, esta serie converge rápidamente a la constante matemática $e - 1$ (ya que la serie de Taylor para $e^x$ evaluada en $x=1$ es $\sum \frac{1}{k!}$ desde $k=0$, y al quitar el término $k=0$ que es $1$, nos queda $e - 1$). Por lo tanto, el costo promedio de verificar si el arreglo está ordenado es casi constante:
$$E[C] \approx e - 1 \approx 1.718$$

In [1]:
import math

def expected_comparisons(n):
    total_sum = 0
    factorial = 1
    # Iteramos desde k=1 hasta n-1
    for k in range(1, n):
        factorial *= k
        total_sum += 1 / factorial
    return total_sum

n = 10
# Comparamos el resultado truncado en n=10 con el límite teórico e-1
print(f"n={n}:  {expected_comparisons(n):.6f}")
print(f"e-1 =  {math.e - 1:.6f}")

n=10:  1.718282
e-1 =  1.718282


### **Enunciado del Problema 6C**
**En la sección 2.3 del paper "Sorting the Slow Way", ¿por qué la variable aleatoria I que cuenta el número de iteraciones del algoritmo tiene distribución geométrica?**

Para entender por qué el número de iteraciones sigue una distribución geométrica, debemos analizar cómo funciona una iteración individual de Bogo-sort:
1. Verifica si el arreglo está ordenado.
2. Si no lo está, realiza un *shuffle* (barajado) completamente aleatorio.

Cada barajado produce una permutación uniformemente aleatoria entre las $n!$ permutaciones posibles. Como solo **una** de esas permutaciones representa el arreglo ordenado correctamente, la probabilidad de éxito en cualquier iteración dada es:
$$p = \frac{1}{n!}$$

La clave aquí es la **independencia**. Cada vez que hacemos un *shuffle*, el algoritmo "olvida" por completo el estado anterior. El resultado del intento actual no se ve afectado por la cantidad de fallos pasados. 

Esto encaja perfectamente con la definición matemática de la **distribución geométrica**, la cual modela la cantidad de ensayos de Bernoulli (experimentos de éxito/fracaso) independientes que se necesitan para obtener el primer éxito. Por lo tanto, definimos que la variable aleatoria $I$ sigue una distribución geométrica:
$$I \sim \text{Geom}\left(\frac{1}{n!}\right)$$

A partir de las propiedades de esta distribución, podemos deducir:
* **Esperanza de iteraciones:** $E[I] = \frac{1}{p} = n!$
* **Esperanza de swaps (intercambios):** Dado que cada *shuffle* de Fisher-Yates realiza $n-1$ intercambios, el total esperado es $E[\text{swaps}] = (n - 1) \cdot n!$

In [3]:
import random
import math

# ═══════════════════════════════════════════════════════
# SIMULACIÓN DEL PROBLEMA 6c: Iteraciones de bogo-sort
# ═══════════════════════════════════════════════════════


def is_sorted(arr):
    for i in range(1, len(arr)):
        if arr[i - 1] > arr[i]:
            return False
    return True

def shuffle_array(arr):
    swaps = 0

    for i in range(len(arr) - 1, 0, -1):
        j = random.randint(0, i)
        arr[i], arr[j] = arr[j], arr[i]
        swaps += 1
    return swaps

def bogo_sort_iterations(n, max_iter=100000):

    arr = list(range(1, n + 1))
    shuffle_array(arr)

    iterations = 0
    swaps = 0
    while not is_sorted(arr):
        swaps += shuffle_array(arr)
        iterations += 1
        if iterations > max_iter:
            return {'iterations': iterations, 'swaps': swaps, 'timed_out': True}
            
    return {'iterations': iterations, 'swaps': swaps, 'timed_out': False}


n = 4 
TRIALS = 5000
total_iter = 0
total_swaps = 0
timed_out_count = 0

for _ in range(TRIALS):
    result = bogo_sort_iterations(n)
    if result['timed_out']:
        timed_out_count += 1
    else:
        total_iter += result['iterations']
        total_swaps += result['swaps']

valid_trials = TRIALS - timed_out_count
n_fact = math.factorial(n)

print(f"n={n} (n!={n_fact}), {TRIALS} ensayos:")
print(f"  E[iteraciones] simulado : {(total_iter / valid_trials):.2f}")
print(f"  E[iteraciones] teórico  : n! = {n_fact}")
print(f"  E[swaps] simulado       : {(total_swaps / valid_trials):.2f}")
print(f"  E[swaps] teórico        : (n-1)·n! = {(n - 1) * n_fact}")

if timed_out_count > 0:
    print(f"  Timeouts: {timed_out_count}")
    
print('\n→ El histograma de iteraciones cae exponencialmente: firma geométrica.')

n=4 (n!=24), 5000 ensayos:
  E[iteraciones] simulado : 23.23
  E[iteraciones] teórico  : n! = 24
  E[swaps] simulado       : 69.68
  E[swaps] teórico        : (n-1)·n! = 72

→ El histograma de iteraciones cae exponencialmente: firma geométrica.
